# Unified rolling-origin evaluation

This notebook compares return-only baselines, Hyndman-Khandakar ARIMA, HAR/HARX, penalized regressions, VAR, and volatility extensions on the full prepared Stage 2 sample. Every model uses the same forecast origins. At each origin, dependent-variable training rows are purged unless `target_end_date <= origin_date`. The target is five-session forward realized variance on Yahoo `GC=F` continuous COMEX gold futures; it is not spot gold. The default `step=5` evaluates the full sample at weekly intervals to limit the cost of repeated ARIMA, GARCH, and penalized fits.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd()
repo_root = next(
    (candidate for candidate in (cwd, *cwd.parents)
     if (candidate / 'pyproject.toml').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Could not locate repository root.')
sys.path.insert(0, str(repo_root / 'src'))

from gold_vol_pipeline.auto_arima import hyndman_khandakar
from gold_vol_pipeline.evaluation import (
    ForecastContext,
    dm_pvalue_matrix,
    forecast_combination,
    qlike,
    rolling_origin_evaluation,
    summary_table,
)
from gold_vol_pipeline.models import (
    arch1_variance_forecast,
    ar1_squared_return_forecast,
    egarch_asym_forecast,
    ewma_estimated_forecast,
    ewma_variance_forecast,
    expanding_variance_forecast,
    garch11_variance_forecast,
    garch_t_forecast,
    garchx_forecast,
    gjr_garch_forecast,
    ma1_squared_return_forecast,
    rolling_variance_forecast,
)
from gold_vol_pipeline.regression_models import (
    PREDICTOR_COLUMNS,
    har_forecast,
    harx_forecast,
    penalized_forecast,
    var_forecast,
)

data_path = repo_root / 'data/processed/stage2_multivariate.csv'
manifest_path = repo_root / 'data/processed/stage2_multivariate_manifest.json'
if not data_path.is_file():
    raise FileNotFoundError('Run the Stage 2 predictor pipeline before this notebook.')
data = pd.read_csv(data_path, parse_dates=['date', 'target_end_date'])
manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.is_file() else {}
horizon = int(manifest.get('horizon', 5))
if manifest.get('target', 'realized_variance') != 'realized_variance':
    raise ValueError('This evaluation requires the realized_variance target.')
required = {'date', 'log_return', 'realized_variance', 'target_end_date', *PREDICTOR_COLUMNS}
missing = sorted(required - set(data.columns))
if missing:
    raise ValueError(f'Stage 2 data is missing required columns: {missing}')
data = data.sort_values('date').reset_index(drop=True)
print(f'Input rows: {len(data):,}; dates: {data.date.min().date()} through {data.date.max().date()}')
print(f'Target: {horizon}-session forward realized variance; model inputs: GC=F return series + Stage 2 predictors')

## Forecast definitions

Model callables receive origin-available return/predictor history and a separately purged target-training frame. The GARCH-X two-step variant is retained as an explicit skipped result because its leakage-safe historical-origin regression design is not implemented. Any failed fit is recorded with `converged=False` and an error message rather than dropped silently.

In [ ]:
def origin_returns(context):
    return context.history.set_index('date')['log_return'].astype(float)

def origin_predictors(context):
    return context.history.set_index('date')[list(PREDICTOR_COLUMNS)].astype(float)

def auto_arima_forecast(context):
    squared = origin_returns(context).to_numpy() ** 2
    fit = hyndman_khandakar(squared, max_p=5, max_q=5, max_d=2, max_order=5)
    return float(fit.forecast(horizon=context.horizon).sum())

models = {
    'historical_variance': lambda c: expanding_variance_forecast(origin_returns(c), horizon=c.horizon),
    'rolling_variance': lambda c: rolling_variance_forecast(origin_returns(c), horizon=c.horizon, window=20),
    'ewma_fixed': lambda c: ewma_variance_forecast(origin_returns(c), horizon=c.horizon, decay=0.94),
    'ewma': lambda c: ewma_estimated_forecast(origin_returns(c), horizon=c.horizon),
    'ar1_squared_return': lambda c: ar1_squared_return_forecast(origin_returns(c), horizon=c.horizon),
    'ma1_squared_return': lambda c: ma1_squared_return_forecast(origin_returns(c), horizon=c.horizon),
    'arch1': lambda c: arch1_variance_forecast(origin_returns(c), horizon=c.horizon),
    'garch11': lambda c: garch11_variance_forecast(origin_returns(c), horizon=c.horizon),
    'auto_arima': auto_arima_forecast,
    'har': lambda c: har_forecast(origin_returns(c), horizon=c.horizon),
    'harx': lambda c: harx_forecast(origin_returns(c), origin_predictors(c), horizon=c.horizon),
    'lasso': lambda c: penalized_forecast(origin_returns(c), origin_predictors(c), horizon=c.horizon, method='lasso'),
    'ridge': lambda c: penalized_forecast(origin_returns(c), origin_predictors(c), horizon=c.horizon, method='ridge'),
    'var': lambda c: var_forecast(origin_returns(c), origin_predictors(c)['vix_close'], horizon=c.horizon),
    'gjr_garch': lambda c: gjr_garch_forecast(origin_returns(c), horizon=c.horizon),
    'egarch_asym': lambda c: egarch_asym_forecast(origin_returns(c), horizon=c.horizon),
    'garch_t': lambda c: garch_t_forecast(origin_returns(c), horizon=c.horizon),
    'garchx': lambda c: garchx_forecast(origin_returns(c), origin_predictors(c), horizon=c.horizon),
    'combination': forecast_combination,
}
MIN_TRAIN = 750
STEP = 5
forecast_path = repo_root / 'data/processed/evaluation_forecasts.csv'
results = rolling_origin_evaluation(
    data,
    models,
    horizon=horizon,
    min_train=MIN_TRAIN,
    step=STEP,
    window='expanding',
    output_path=forecast_path,
)
print(f'Wrote {len(results):,} forecast/model rows to {forecast_path}')
display(results.groupby(['model', 'converged']).size().rename('rows').to_frame())

In [ ]:
summary = summary_table(results, benchmark='garch11', horizon=horizon, top_n=5)
display(summary)
dm_mse = summary.attrs['dm_pvalue_matrix_mse']
dm_qlike = summary.attrs['dm_pvalue_matrix_qlike']
display(dm_qlike.style.format('{:.3g}'))

In [ ]:
plot_dir = repo_root / 'plots/final_evaluation'
plot_dir.mkdir(parents=True, exist_ok=True)
valid_summary = summary.loc[summary['n'] > 0]
best_models = valid_summary['model'].head(3).tolist()
plot_models = list(dict.fromkeys([*best_models, 'garch11']))

# Actual versus forecast for the three lowest-MSE models and GARCH(1,1).
wide = results[results.model.isin(plot_models)].pivot(index='origin_date', columns='model', values='forecast')
actual = results[results.model == 'garch11'].set_index('origin_date')['actual']
figure, axis = plt.subplots(figsize=(13, 6))
axis.plot(actual.index, actual, label='Observed', color='black', linewidth=1.5)
for name in plot_models:
    if name in wide:
        axis.plot(wide.index, wide[name], label=name, alpha=0.85)
axis.set_title('5-session forward realized variance, observed vs forecast')
axis.set_xlabel('Forecast origin date')
axis.set_ylabel('Realized variance (squared log-return units; 5 sessions)')
axis.legend(ncol=2)
figure.tight_layout()
figure.savefig(plot_dir / 'actual_vs_forecast.png', dpi=200)
plt.close(figure)

# Cumulative QLIKE loss difference: negative values favor the candidate.
benchmark_rows = results[results.model == 'garch11'][['origin_date', 'actual', 'forecast']].rename(columns={'forecast': 'benchmark'})
figure, axis = plt.subplots(figsize=(13, 6))
for name in plot_models:
    if name == 'garch11':
        continue
    model_rows = results[results.model == name][['origin_date', 'actual', 'forecast']]
    aligned = model_rows.merge(benchmark_rows, on=['origin_date', 'actual'], validate='one_to_one').dropna()
    if len(aligned) and (aligned[['actual', 'forecast', 'benchmark']] > 0).all().all():
        model_loss, _ = qlike(aligned.actual, aligned.forecast)
        benchmark_loss, _ = qlike(aligned.actual, aligned.benchmark)
        axis.plot(aligned.origin_date, (model_loss - benchmark_loss).cumsum(), label=name)
axis.axhline(0, color='black', linewidth=0.8)
axis.set_title('Cumulative QLIKE difference vs GARCH(1,1)')
axis.set_xlabel('Forecast origin date')
axis.set_ylabel('Cumulative QLIKE loss difference (candidate − benchmark; 5 sessions)')
axis.legend()
figure.tight_layout()
figure.savefig(plot_dir / 'cumulative_qlike_vs_garch11.png', dpi=200)
plt.close(figure)

# Selected lasso coefficients across origins, with selection frequency.
lasso_rows = results[(results.model == 'lasso') & results.coefficients.notna()].copy()
coefficient_rows = []
for row in lasso_rows.itertuples():
    for feature, coefficient in json.loads(row.coefficients).items():
        coefficient_rows.append({'origin_date': row.origin_date, 'feature': feature, 'coefficient': coefficient})
coefficient_frame = pd.DataFrame(coefficient_rows)
figure, axes = plt.subplots(2, 1, figsize=(13, 9), gridspec_kw={'height_ratios': [2, 1]})
if not coefficient_frame.empty:
    frequencies = coefficient_frame.groupby('feature').origin_date.nunique().div(lasso_rows.origin_date.nunique()).sort_values(ascending=False)
    selected_features = frequencies.head(8).index.tolist()
    for feature in selected_features:
        path = coefficient_frame[coefficient_frame.feature == feature]
        axes[0].plot(path.origin_date, path.coefficient, label=feature)
    axes[0].legend(ncol=2)
    axes[1].bar(frequencies.head(12).index, frequencies.head(12).values)
    axes[1].tick_params(axis='x', rotation=45)
axes[0].set_title('Lasso selected coefficient paths across origins')
axes[0].set_ylabel('Log-variance coefficient (training-standardized predictors; 5 sessions)')
axes[0].set_xlabel('Forecast origin date')
axes[1].set_title('Lasso coefficient selection frequency (5-session target)')
axes[1].set_ylabel('Fraction of fitted origins')
axes[1].set_xlabel('Feature')
figure.tight_layout()
figure.savefig(plot_dir / 'lasso_coefficients.png', dpi=200)
plt.close(figure)

# Pairwise DM QLIKE p-values for the top models.
figure, axis = plt.subplots(figsize=(8, 7))
image = axis.imshow(dm_qlike.to_numpy(dtype=float), vmin=0, vmax=1, cmap='viridis_r')
axis.set_xticks(range(len(dm_qlike.columns)), dm_qlike.columns, rotation=45, ha='right')
axis.set_yticks(range(len(dm_qlike.index)), dm_qlike.index)
axis.set_title('Pairwise Diebold-Mariano p-values (QLIKE; 5-session horizon)')
figure.colorbar(image, ax=axis, label='Two-sided p-value')
figure.tight_layout()
figure.savefig(plot_dir / 'dm_qlike_heatmap.png', dpi=200)
plt.close(figure)
print(f'Saved 200 dpi figures to {plot_dir}')

## Model-comparison handoff

At the 5% level under MSE, models beating the GARCH(1,1) benchmark after Holm correction: **<fill after running>** (adjusted p-values: **<fill after running>**). Under QLIKE: **<fill after running>** (adjusted p-values: **<fill after running>**).

Interpret these tests alongside the loss magnitudes, forecast calibration, overlapping-target dependence, failed-fit counts, and source-timing limitations. No model-selection decision is implied until the team reviews this evidence.